> **Ported notebook.** This notebook was originally a ``midas-ff-pipeline`` walkthrough. ``midas-ff-pipeline`` is deprecated as of 0.4.0; the FF workflow now lives in ``midas-pipeline`` with ``--scan-mode ff``. All API + CLI calls below have been retargeted; behaviour is unchanged. The legacy package will be removed in 1.0.0.


# Multi-detector pinwheel demo

Drives the four-panel hydra pinwheel layout (Lsd ≈ 3.3 m, BC ≈ (2300, 2150),
tx ≈ 297° / 27° / 117° / 207°) end-to-end. Each panel produces its own
peak list; the `cross_det_merge` stage concatenates them with a side-car
`Spots_det.bin` so downstream stages see a single global `Spots.bin`.
Per-(ring, η) coverage is emitted into the merged `paramstest.txt` so
the indexer's completeness denominator only counts predicted spots that
actually fall on a panel.

Synthetic data is generated by `midas_diffract.HEDMForwardModel` with
`multi_mode="panel"` and the FF tilt inverse.

**Edit the paths in cell 3, then *Run All*.**

In [ ]:
from pathlib import Path
import json
import shutil

import matplotlib.pyplot as plt
import numpy as np

from midas_pipeline import Pipeline, PipelineConfig, DetectorConfig, ScanGeometry
from midas_pipeline.testing import generate_pinwheel_synthetic_dataset


## 1. Inputs — edit these

In [ ]:
work_dir = Path('/tmp/midas_pipeline_multidet')

n_grains = 100
n_panels = 4         # hydra-real: 4 panels at tx 297/27/117/207
n_cpus = 8
device = 'cpu'       # 'cuda' or 'cuda:0' on a GPU box
dtype = 'float64'


## 2. Generate the pinwheel forward simulation

`generate_pinwheel_synthetic_dataset` writes one `.MIDAS.zip` per panel
plus a `detectors.json` ready for `Pipeline(detectors=…)`. Hydra-real
geometry is on by default (`use_hydra_geometry=True`).

In [ ]:
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

zips, detectors_json = generate_pinwheel_synthetic_dataset(
    out_dir=work_dir / 'sim',
    n_grains=n_grains,
    seed=42,
    n_panels=n_panels,
    use_hydra_geometry=True,
    device=device,
)
for z in zips:
    print(f'  panel zarr: {z}')
print(f'detectors.json: {detectors_json}')

## 3. Inspect the per-panel geometry

In [ ]:
detectors = DetectorConfig.load_many(detectors_json)
for d in detectors:
    print(f'  det {d.det_id}: Lsd={d.lsd:.0f}μm  BC=({d.y_bc:.1f}, {d.z_bc:.1f})  '
          f'tx={d.tx:.1f}°  ty={d.ty:.2f}°  tz={d.tz:.2f}°')


## 4. Run the pipeline

The pipeline auto-routes: each panel goes through `hkl → peakfit → merge
→ calc_radius → transforms` in its own subdir, then `cross_det_merge`
concatenates everything before `global_powder → binning → indexing →
refinement → process_grains`. Multi-det runs auto-switch refinement loss
from `pixel` to `angular` — see `stages/refine.py`.

In [ ]:
params_for_pipeline = next(zips[0].parent.glob('*.txt'))    # the per-panel Parameters.txt
pipe = Pipeline(
    config=PipelineConfig(
    scan=ScanGeometry.ff(),
        result_dir=str(work_dir / 'run'),
        params_file=str(params_for_pipeline),
        detectors_json=str(detectors_json),
        n_cpus=n_cpus,
        device=device,
        dtype=dtype,
    ),
    detectors=detectors,
)
pipe.run()
result = pipe.layer_result

## 5. Per-stage timings

`all_stage_results()` returns the chronologically-ordered StageResult
list (zip_convert / consolidation are skipped here because the zarrs are
already produced and `--generate-h5` isn't set).

In [ ]:
print(f'Total grains:    {result.n_grains}')
print(f'Total runtime:   {result.total_duration_s():.1f}s')
print()
for stage in result.all_stage_results():
    print(f'  {stage.stage_name:<22s} {stage.duration_s:7.2f}s')

## 6. Per-detector peak counts

In [ ]:
merge = result.cross_det_merge
print(f'total spots after cross-det merge: {merge.n_total_spots:,}')
for det, n in zip(detectors, merge.n_per_detector):
    print(f'  det {det.det_id}: {n:,}')

## 7. Spots.bin coloured by det_id

`Spots_det.bin` is a parallel int32 array — one entry per row of
`Spots.bin`, holding the source panel id.

In [ ]:
spots = np.fromfile(pipe.layer_dir / 'Spots.bin', dtype=np.float64).reshape(-1, 9)
det_id = np.fromfile(pipe.layer_dir / 'Spots_det.bin', dtype=np.int32)

fig, ax = plt.subplots(figsize=(7, 7))
for d in detectors:
    sel = det_id == d.det_id
    ax.scatter(spots[sel, 0], spots[sel, 1], s=2, alpha=0.5, label=f'det {d.det_id}')
ax.set_aspect('equal')
ax.set_xlabel('Y_lab (μm)')
ax.set_ylabel('Z_lab (μm)')
ax.legend()
ax.set_title(f'Multi-det Spots.bin — {spots.shape[0]:,} obs')
plt.tight_layout()
plt.show()

## 8. Final grain map

In [ ]:
df = result.grains_df()
print(f'{len(df)} grains')
df.head()